# Solutions — Props & component communication

Only look here after you've actually tried the exercises in `props.ipynb`.

### LESSON 14 — Exercise

In [ ]:
function l14rowText({ description, time }) {
  return `${description} — ${time}`;
}

const l14activity = [
  { description: "Order #4821 shipped", time: "2 hours ago" },
  { description: "Invoice #1190 paid", time: "4 hours ago" },
  { description: "Refund issued for order #4790", time: "yesterday" },
  { description: "New customer: Baltic Foods", time: "yesterday", unread: true },
  { description: "Stock alert: Blue mugs below 20 units", time: "2 days ago" },
];

for (const item of l14activity) {
  console.log(l14rowText(item));
}

// Part 3 — the extra `unread` property causes no trouble because destructuring only
// pulls out the names you ask for. Everything else stays in the object, unread (in both
// senses). For the row to respond to it, the function would have to ask for it:
//
//   function l14rowText({ description, time, unread }) { ... }
//
// Not asking is the normal case. A component reads the props it needs and ignores the rest.

**Common mistakes.**

- `function l14rowText(description, time)` — two parameters instead of one object. It works
  if you call it as `l14rowText(item.description, item.time)`, but it is not how React calls
  a component. React always passes **one** object, so the signature has to accept one.
- Forgetting the braces: `function l14rowText(item)` and then writing `${description}` in
  the body throws `ReferenceError: description is not defined` — there is no such variable,
  only `item.description`. Either destructure, or read it off the object.
- Destructuring a name that does not exist — `{ text, time }` instead of
  `{ description, time }` — throws nothing at all. `text` is simply `undefined`, and the
  line prints `undefined — 2 hours ago`. That silence is the most common props bug there
  is, and it is why a missing value shows up as the word "undefined" on the page rather
  than as an error.

### LESSON 14 — Mini challenge

**Steps 1 and 2 — the bug.** The function edits the object it was handed, so the second call
starts from an already-modified label, and the caller's object is changed for good.

In [ ]:
const l14card = { label: "Revenue", value: "128,400 EUR" };

function l14withCurrency(card) {
  card.label = card.label + " (EUR)";
  return `${card.label} — ${card.value}`;
}

console.log(l14withCurrency(l14card));
console.log(l14withCurrency(l14card)); // " (EUR)" again — the damage accumulates
console.log(l14card);                  // the ORIGINAL object, permanently altered

**Step 3 — the fix.** Read the props, build a new string, change nothing. Called any number
of times, the answer is the same and the caller's object is untouched.

In [ ]:
const l14cardFixed = { label: "Revenue", value: "128,400 EUR" };

function l14withCurrencyFixed({ label, value }) {
  return `${label} (EUR) — ${value}`;
}

console.log(l14withCurrencyFixed(l14cardFixed));
console.log(l14withCurrencyFixed(l14cardFixed)); // identical
console.log(l14cardFixed);                       // untouched

**Step 4 — why immutability matters.** The parent did not give the child a copy. It gave it
a reference to the very object it is still holding, and probably still using — to render
something else, to compare against, to pass to another component.

Two different mistakes hide under "changing props", and they behave nothing alike.

**Assigning to the props object itself** — `props.label = "…"` — does not even get off the
ground in development. React freezes every element's props, so the assignment throws
immediately:

```text
TypeError: Cannot assign to read only property 'label' of object '#<Object>'
```

That freeze is development-only. A production build does not freeze, and there the same line
is silent — which is worth knowing, because it means this is a mistake your dev environment
catches for you rather than one you can rely on the language to prevent.

**Mutating an object or array that arrived through a prop** — which is exactly what this
exercise does — is the dangerous one. React freezes the props object, not the values inside
it. So the child is reaching up and editing the parent's own data from below: nothing warns
you, the parent has no idea it happened, and the bug surfaces somewhere else entirely,
usually the second time something renders.

That is why React states the rule flatly rather than as advice: **don't try to "change
props"**. A component that needs different props asks its parent for them. Topic 9 is where
the parent gains the ability to say yes.

### LESSON 15 — Exercise

In [ ]:
function l15summary({ label, value, unit = "EUR", featured = false }) {
  const shown = typeof value === "number" ? value.toLocaleString("en-US") : value;
  const prefix = featured ? "* " : "";
  return `${prefix}${label}: ${shown} ${unit}`;
}

console.log(l15summary({ label: "Revenue", value: 128400 }));
console.log(l15summary({ label: "Revenue", value: "128,400" }));
console.log(l15summary({ label: "Orders", value: 1284, unit: "items", featured: true }));
console.log(l15summary({ label: "Refunds", value: 37, unit: null }));
console.log(l15summary({ label: "Visitors", value: 612, unit: undefined }));

// Refunds passes `unit: null` and Visitors passes `unit: undefined`. Only Visitors gets
// "EUR". A destructuring default fires when the value is undefined — which covers both a
// missing prop and an explicit undefined — and null is a real value, so it is kept.

**Common mistakes.**

- Testing the value with `if (value)` instead of `typeof value === "number"`. A `value` of
  `0` is falsy, so a genuine zero would be treated as a string and skip formatting.
- Writing `unit = unit || "EUR"` inside the function instead of a destructuring default.
  That *looks* more robust and is worse: it also replaces `""` and `0`, so a deliberate
  empty unit silently becomes `"EUR"`. If you want to catch `null` as well as `undefined`,
  `unit ?? "EUR"` says exactly that and nothing more.
- Formatting with `.toLocaleString()` and no locale. The grouping separator then depends on
  the machine the code runs on — the same code prints `128,400` on one and `128.400` on
  another. Passing `"en-US"` makes the output the same everywhere.

### LESSON 15 — Mini challenge

```jsx
<PriceTag amount="19.99" currency={EUR} discount=0 featured="false" />
```

**1. `amount="19.99"` — a string, not a number.** Quotes always produce a string, so any
arithmetic downstream misbehaves: `amount * 2` gives `39.98` because JavaScript coerces,
but `amount + 1` gives `"19.991"`. *When you find out:* possibly never. It renders
correctly, and the bug appears the first time someone adds to it.
Fix: `amount={19.99}`.

**2. `currency={EUR}` — braces mean "evaluate this JavaScript".** `EUR` is read as a
variable name, not as text. Unless something called `EUR` exists, the browser throws
`ReferenceError: EUR is not defined`. *When you find out:* at runtime, in the console —
the same failure shape as the renamed component in LESSON 4.
Fix: `currency="EUR"`.

**3. `discount=0` — a bare value with no quotes and no braces.** JSX allows exactly two
forms for an attribute value: a quoted string, or a braced expression. A bare number is
neither, so the file does not compile:

```text
[PARSE_ERROR] Unexpected token
```

*When you find out:* immediately, at build time. Nothing loads.
Fix: `discount={0}`.

**4. `featured="false"` — the string `"false"`.** It is a non-empty string, so it is
**truthy**. The component is featured, permanently, and the code reads as though it says the
opposite. *When you find out:* quietly, by noticing the UI is wrong.
Fix: `featured={false}` — or simply leave the prop off.

**The corrected tag:**

```jsx
<PriceTag amount={19.99} currency="EUR" discount={0} featured={false} />
```

**One thing worth noticing about the order.** Mistake 3 stops the build, so while it is
there you cannot see mistakes 1, 2 or 4 at all — exactly the situation from LESSON 3, where
the missing export hid the container error. Fix what stops the build, reload, and look
again.

### LESSON 16 — Exercise

`l16handleSelect` and `l16makeRow` are repeated here so this notebook runs on its own.

In [ ]:
function l16handleSelect(id) {
  console.log("  parent was told about", id);
}

function l16makeRow({ label, id, onSelect }) {
  return { label, onSelect, select: () => onSelect(id) };
}

// Part 1
const l16activity = [
  { label: "Order #4821 shipped", id: 4821 },
  { label: "Invoice #1190 paid", id: 1190 },
  { label: "Refund for order #4790", id: 4790 },
];

const l16rows = l16activity.map((item) =>
  l16makeRow({ label: item.label, id: item.id, onSelect: l16handleSelect }),
);

console.log("three rows built, nothing reported yet");
l16rows[0].select();
l16rows[2].select();
// 1190 is never mentioned — that row was built but never acted on.

**Part 2 — pass versus call.** The second row reports `999` while it is being *built*, which
is the first clue that something is wrong. What it stores is the return value of the call.

In [ ]:
const l16good = l16makeRow({ label: "good", id: 1, onSelect: l16handleSelect });

console.log("--- building the bad row ---");
const l16bad = l16makeRow({ label: "bad", id: 2, onSelect: l16handleSelect(999) });
console.log("--- built ---");

// Inspect what each row actually stored. These read the rows themselves — nothing is
// invoked again, and nothing is assumed.
console.log("good row stored a", typeof l16good.onSelect);
console.log("bad row stored a", typeof l16bad.onSelect);

l16good.select();

try {
  l16bad.select();
} catch (error) {
  console.log("bad row throws:", error.constructor.name + ":", error.message);
}

// `l16handleSelect(999)` ran immediately and returned undefined, so the row stored
// undefined as its callback. Calling it later is calling undefined — hence
// "TypeError: onSelect is not a function".

**Common mistakes.**

- Expecting the bad row to fail at the moment it is built. It does not — storing `undefined`
  is perfectly legal. The failure is deferred to the moment somebody acts on the row, which
  in a real app means a user clicking something long after the code that caused it ran.
- Reading the early `parent was told about 999` as "it worked". It is the opposite: that
  output is the function being used up during render instead of being handed over.
- Writing `onSelect: () => l16handleSelect` — an arrow that *returns* the function rather
  than calling it. `onSelect(id)` then returns the function and reports nothing.

### LESSON 16 — Mini challenge

**1. Both symptoms, one cause.** Braces evaluate their contents immediately (LESSON 8), so
`handleSelect(4821)` is a **call**, not a reference. React's docs put it plainly: the `()`
"fires the function immediately during rendering, without any clicks".

- *The early output* is that call running while the row is being rendered.
- *The error on click* is the consequence: the call returned `undefined`, so `undefined` is
  what was passed as `onSelect`. When the click finally does `onSelect(id)`, it is calling
  `undefined` — `TypeError: onSelect is not a function`.

One mistake, two symptoms that look unrelated, separated in time by however long it takes
someone to click.

**2. Where the id should come from.** The child. `ActivityRow` already receives `id` as a
prop — it is rendering that row and knows exactly which one it is. The parent should hand
over the function and nothing else.

**3. The corrected pair.**

```jsx
{/* parent */}
<ActivityRow label="Order #4821" id={4821} onSelect={handleSelect} />
```

```jsx
{/* inside ActivityRow */}
<button onClick={() => onSelect(id)}>{label}</button>
```

**4. Why the child should supply it.** Because the parent writes the tag once and the child
renders every row. Baking `4821` into the prop means the parent needs a different function —
or a different tag — for every single row, and it has to know each id in advance. Handing
down one `handleSelect` and letting each row report itself scales to three rows or three
hundred without the parent changing at all.

That is the shape of nearly every callback prop you will write: **the parent supplies the
behaviour, the child supplies the specifics.**

> The arrow inside `onClick` is doing something worth a second look — it delays the call so
> that `onSelect(id)` happens on click rather than during render. Topic 8 comes back to that
> properly once events are the subject.

### LESSON 17 — Exercise

This one is done in `playground/src/experiments/02-props.jsx`, so the answers are written
out rather than run.

**1. `Panel` with and without a header.**

```jsx
<Panel header={<h2>Revenue</h2>}>
  <p>12,400 EUR</p>
</Panel>

<Panel>
  <p>This panel has no header.</p>
</Panel>
```

The second renders fine. `header` is `undefined`, and `undefined` renders nothing
(LESSON 10) — so `<div>{header}</div>` produces an empty `<div>` and the page moves on.
You wrote no condition, and there was none to write.

**2. A different header.** Anything works, because `Panel` never looks at it:

```jsx
<Panel header={<input placeholder="Search orders" />}>
  <p>No results yet.</p>
</Panel>
```

That is the whole argument for slots in one line. A flag-based `Panel` would have needed a
`searchable` prop, a `placeholder` prop, and a decision inside it. This needed nothing.

**3. `Media` with a component prop.**

```jsx
<Media Icon={StarIcon}>Featured</Media>
```

Both icons appear, because `Media` renders `<Icon />` twice. Note what that means: the
caller handed over a component and had no say in how often it was used. With an element —
`icon={<StarIcon />}` — the caller would have handed over one already-built icon, and
rendering it twice would mean rendering the same element in two places.

**4. Lowercasing it.** Change one `<Icon />` to `<icon />` and the star disappears from that
position. In the elements panel you find an empty `<icon></icon>`, and the console repeats
LESSON 4's warning:

```text
The tag <icon> is unrecognized in this browser.
If you meant to render a React component, start its name with an uppercase letter.
```

The variable `Icon` really does hold your component. It makes no difference: JSX decides
what a tag means from its **first letter**, before React sees anything. `<icon />` compiles
to `jsx("icon", …)` — a string, an HTML tag name — and your component is never called. The
rule you met for `<greeting />` in LESSON 4 applies just as hard to a prop.

**Common mistakes.**

- Writing `<Panel header="Revenue">`. That works, and it is not a slot — it is a string prop
  (LESSON 15). Fine if the header is always plain text; the moment one screen wants a button
  there, you need the element form.
- Destructuring the component prop as `{ icon }` and then writing `<icon />`. Same failure
  as step 4, and harder to spot because everything looks consistent. If a prop holds a
  component, name it with a capital: `{ Icon }`.
- Passing `icon={StarIcon()}` — calling it. That is LESSON 16's mistake in new clothes: you
  hand over the *result* instead of the component. It happens to work here, because the
  result is an element, but it is no longer a component prop and the child can no longer
  decide anything about it.

### LESSON 17 — Mini challenge

| # | API | Answer | Why |
|---|---|---|---|
| 1 | `<Button loading />` | **flag** | Two states, and the button owns both. The caller has no opinion about what a loading button looks like — that is the button's job, and it should look the same everywhere |
| 2 | `<Card footer={…} />` | **slot** | The variation is unbounded and belongs entirely to the caller. Every footer added as a flag would be a prop plus a decision inside `Card` |
| 3 | `<Dialog title="Delete order?" />` | **neither — a plain string prop is right** | It varies, but only as *text*. A slot here buys nothing and costs the caller `title={<span>Delete order?</span>}` at every call site |
| 4 | `<Table emptyState={…} />` | **slot** | Same as 2: different markup per table, and `Table` has no business knowing about any of it |

**`Icon={StarIcon}` versus `icon={<StarIcon />}`.**

The first passes the **component** — a function. `Media` decides when and how often to call
it, and it calls it twice. The second passes an **element** — one already-built object
describing one star.

You need the first whenever the child controls the rendering: it wants the icon in two
places, or inside something only it knows about. You want the second when the caller has
already decided exactly what that icon is and the child should just place it.

**Why number 3 is the important one.** The lesson makes slots look strictly better, and this
is the counterexample. "It varies" is not the test. The test is *what kind* of variation:
text varying is a value, markup varying is a slot, and a closed set of looks the component
owns is a flag. Turning every prop into a slot is its own kind of mess — every call site
gets longer, and simple things stop looking simple.